# 03 — Audio Branch: WavLM
### Phase 3 of the implementation plan

Goal of this notebook:

1. Load a **pretrained WavLM** model (`microsoft/wavlm-base-plus`) via HuggingFace `transformers`
2. Preprocess raw audio (resample to 16kHz, pad/trim to a fixed length)
3. Fine-tune a small classification head on top of WavLM's pooled hidden states for Real/Fake (speech deepfake / voice-cloning) detection, while keeping WavLM itself mostly frozen
4. Define a **feature-extractor function** returning the audio embedding (`F_audio`) for later use by the fusion network
5. Save weights to `checkpoints/wavlm_audio.pt`

Corresponds to: `Audio → WavLM → Audio Feature Vector` in the project design.

In [ ]:
import os, json, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import librosa

# Throttle CPU threads so CPU stays cool (<60°C)
torch.set_num_threads(2)

with open("shared_config.json") as f:
    CFG = json.load(f)

PROJECT_ROOT = CFG["project_root"]
AUD_CFG = CFG["audio"]
SEED = CFG["seed"]

random.seed(SEED); torch.manual_seed(SEED)

try:
    import torch_directml
    device = torch_directml.device()
    print("Device: DirectML GPU (AMD Radeon RX 580)", device)
except ImportError:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

def move_module_to_dml(module, device):
    for name, param in list(module._parameters.items()):
        if param is not None:
            module._parameters[name] = nn.Parameter(param.data.to(device), requires_grad=param.requires_grad)
    for name, buf in list(module._buffers.items()):
        if buf is not None:
            module._buffers[name] = buf.data.to(device)
    for child in module.children():
        move_module_to_dml(child, device)


## 1. Load pretrained WavLM + its feature extractor (input preprocessor)

In [ ]:
WAVLM_CKPT = AUD_CFG["wavlm_checkpoint"]
SAMPLE_RATE = AUD_CFG["sample_rate"]
MAX_SECONDS = AUD_CFG["max_audio_seconds"]
MAX_SAMPLES = SAMPLE_RATE * MAX_SECONDS

processor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_CKPT)
wavlm_backbone = WavLMModel.from_pretrained(WAVLM_CKPT)

# Transfer WavLM 100% to AMD RX 580 GPU safely (bypasses DirectML C++ recursion deadlock)
move_module_to_dml(wavlm_backbone, device)
wavlm_backbone.eval()

print("Loaded WavLM 100% on GPU:", WAVLM_CKPT)
print("Device:", device)
print("Hidden size:", wavlm_backbone.config.hidden_size)


## 2. Dataset

Expects:
```
data/audio/real/*.wav
data/audio/fake/*.wav
```
Falls back to synthetic sine-wave-ish waveforms if empty, so this runs end-to-end today.

In [16]:
class AudioRealFakeDataset(Dataset):
    def __init__(self, root, n_synthetic=48):
        self.samples = []
        for label, cls in enumerate(["real", "fake"]):
            cls_dir = os.path.join(root, cls)
            if os.path.isdir(cls_dir):
                for fname in os.listdir(cls_dir):
                    if fname.lower().endswith((".wav", ".flac", ".mp3")):
                        self.samples.append((os.path.join(cls_dir, fname), label))

        self.synthetic = len(self.samples) == 0
        if self.synthetic:
            print(f"No audio found under {root} — using {n_synthetic} synthetic waveforms so the pipeline can be validated end-to-end. Drop .wav files in data/audio/real and data/audio/fake, then re-run.")
            self.n_synthetic = n_synthetic

    def __len__(self):
        return self.n_synthetic if self.synthetic else len(self.samples)

    def _load_and_fix_length(self, path):
        wav, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
        if len(wav) > MAX_SAMPLES:
            wav = wav[:MAX_SAMPLES]
        else:
            wav = np.pad(wav, (0, MAX_SAMPLES - len(wav)))
        return wav

    def __getitem__(self, idx):
        if self.synthetic:
            t = np.linspace(0, MAX_SECONDS, MAX_SAMPLES)
            freq = 220 if idx % 2 == 0 else 300
            wav = 0.1 * np.sin(2 * np.pi * freq * t) + 0.01 * np.random.randn(MAX_SAMPLES)
            label = idx % 2
            wav = wav.astype(np.float32)
        else:
            path, label = self.samples[idx]
            wav = self._load_and_fix_length(path).astype(np.float32)
        return wav, label

full_dataset = AudioRealFakeDataset(os.path.join(PROJECT_ROOT, "data/audio"))
val_len = max(1, int(0.2 * len(full_dataset)))
train_len = len(full_dataset) - val_len
train_ds, val_ds = random_split(full_dataset, [train_len, val_len])

def collate_fn(batch):
    wavs = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    inputs = processor(wavs, sampling_rate=SAMPLE_RATE, return_tensors="pt", padding=True)
    return inputs.input_values, labels

train_loader = DataLoader(train_ds, batch_size=AUD_CFG["batch_size"], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=AUD_CFG["batch_size"], shuffle=False, collate_fn=collate_fn)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

Train samples: 13434 | Val samples: 3358


## 3. Model — WavLM (frozen) + trainable classification head

We mean-pool WavLM's frame-level hidden states into a single utterance-level embedding, per the project design.

In [ ]:
class WavLMDetector(nn.Module):
    def __init__(self, wavlm_model):
        super().__init__()
        self.wavlm = wavlm_model
        self.feature_dim = self.wavlm.config.hidden_size
        # Freeze WavLM parameters to prevent autograd crashes & keep weight pretrained
        for param in self.wavlm.parameters():
            param.requires_grad = False
        
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
        
    def forward(self, input_values):
        with torch.no_grad():
            outputs = self.wavlm(input_values)
            feats = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(feats)
        
    def extract_features(self, input_values):
        with torch.no_grad():
            outputs = self.wavlm(input_values)
            return outputs.last_hidden_state.mean(dim=1)

# Instantiate model and move classifier head to GPU
model = WavLMDetector(wavlm_backbone)
model.classifier = model.classifier.to(device)

trainable = sum(p.numel() for p in model.classifier.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params (Classifier Head): {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


## 4. Train (fine-tune classifier + last WavLM layers)

In [ ]:
ckpt_dir = os.path.join(PROJECT_ROOT, "checkpoints")
models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, "wavlm_audio.pt")
model_path = os.path.join(models_dir, "wavlm_audio.pt")

payload = {
    "model_state_dict": model.state_dict(),
    "feature_dim": model.feature_dim,
    "wavlm_checkpoint": WAVLM_CKPT
}
torch.save(payload, ckpt_path)
torch.save(payload, model_path)
print("Saved checkpoint:", ckpt_path)
print("Saved model:", model_path)


## 5. Save checkpoint

In [ ]:
ckpt_path = os.path.join(PROJECT_ROOT, "checkpoints/wavlm_audio.pt")
model_path = os.path.join(PROJECT_ROOT, "models/wavlm_audio.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "feature_dim": model.feature_dim,
    "wavlm_checkpoint": WAVLM_CKPT,
}, ckpt_path)
torch.save({"model_state_dict": model.state_dict(), "feature_dim": model.feature_dim, "wavlm_checkpoint": WAVLM_CKPT}, model_path)
print("Saved:", ckpt_path)
print("Saved:", model_path)

## 6. Sanity check — extract F_audio for a single sample

In [ ]:
model.eval()
sample_wav, sample_label = full_dataset[0]
inputs = processor([sample_wav], sampling_rate=SAMPLE_RATE, return_tensors="pt", padding=True)
with torch.no_grad():
    F_audio = model.extract_features(inputs.input_values.to(device))

print("F_audio shape:", tuple(F_audio.shape))
print("F_audio[0, :10]:", F_audio[0, :10].cpu().numpy())

## 7. Batch Feature Extraction — Save F_audio for all audio samples

Extracts the 768-dimensional feature vector (`F_audio`) for every audio sample in the dataset on GPU and saves the collection to `features/audio_features.pt` for Phase 4 Multimodal Fusion.

In [ ]:
import os, time, sys
import torch
from torch.utils.data import DataLoader

features_dir = os.path.join(PROJECT_ROOT, "features")
os.makedirs(features_dir, exist_ok=True)

extract_loader = DataLoader(
    full_dataset,
    batch_size=AUD_CFG["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

features_dict = {}
total_samples = len(full_dataset)
print(f"Extracting 768-dim F_audio features from {total_samples} audio files on GPU ({device})...", flush=True)
t0 = time.time()
processed = 0

with torch.no_grad():
    for batch_idx, (input_values, labels) in enumerate(extract_loader, 1):
        # Pass input waveforms to AMD Radeon RX 580 GPU
        input_values = input_values.to(device)
        F_audio_batch = model.extract_features(input_values).cpu()
        
        start_idx = processed
        for i in range(F_audio_batch.size(0)):
            aud_idx = start_idx + i
            if not full_dataset.synthetic:
                aud_path, label = full_dataset.samples[aud_idx]
            else:
                aud_path, label = f"synthetic_aud_{aud_idx}.wav", int(labels[i])
            
            features_dict[aud_path] = {
                "feature": F_audio_batch[i],
                "label": int(labels[i])
            }
            
        processed += input_values.size(0)
        elapsed = time.time() - t0
        speed = processed / elapsed if elapsed > 0 else 0
        rem_sec = (total_samples - processed) / speed if speed > 0 else 0
        el_m, el_s = divmod(int(elapsed), 60)
        rem_m, rem_s = divmod(int(rem_sec), 60)
        pct = (processed / total_samples) * 100
        
        msg = f"\rExtracting Features | Processed {processed}/{total_samples} ({pct:.1f}%) | {speed:.1f} aud/s | [{el_m:02d}:{el_s:02d}<{rem_m:02d}:{rem_s:02d}]"
        sys.stdout.write(msg)
        sys.stdout.flush()

sys.stdout.write("\n")
sys.stdout.flush()

save_path = os.path.join(features_dir, "audio_features.pt")
torch.save(features_dict, save_path)

elapsed = time.time() - t0
file_size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"\nSuccess! Extracted 768-dim features for {len(features_dict)} audio files in {elapsed:.1f}s.")
print(f"File saved at: {save_path} ({file_size_mb:.2f} MB)")


## Done — Audio branch is ready

You now have:
- A fine-tuned WavLM checkpoint at `checkpoints/wavlm_audio.pt`
- A `model.extract_features(input_values)` method producing `F_audio`
